In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options  # ChromeOptions 클래스를 import 해야 합니다.

# WebDriver 서비스 설정
chrome_driver_path = r"C:/Users/kfq/Downloads/chromedriver-win64/chromedriver.exe"
service = Service(executable_path=chrome_driver_path)

# ChromeOptions 설정 (여기서는 기본 옵션을 사용합니다. 필요에 따라 추가 옵션을 설정할 수 있습니다.)
options = Options()

# WebDriver 인스턴스 생성
driver = webdriver.Chrome(service=service, options=options)

# 웹사이트 열기 예제
driver.get("https://map.kakao.com/")

# 브라우저 닫기 (작업 후 필요에 따라 호출)
# driver.quit()


In [ ]:
from selenium import webdriver as wb
from bs4 import BeautifulSoup as bs
import time
from selenium.webdriver.common.by import By

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\merged_data_cleaned.csv'
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_geocoded전체.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder")

# 지오코딩 함수 정의
def geocode_address(address):
    try:
        location = geolocator.geocode(address)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except GeocoderTimedOut:
        return geocode_address(address)

# 위도와 경도 컬럼 추가
df['위도'], df['경도'] = zip(*df['도로명 주소'].apply(geocode_address))

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_combined_unique_final.csv'
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_geocoded서울과경기.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder", timeout=10)  # timeout 값을 10초로 설정

# 지오코딩 함수 정의
def geocode_address(address, retries=5):
    for attempt in range(retries):
        try:
            location = geolocator.geocode(address)
            if location:
                return location.latitude, location.longitude
            else:
                return None, None
        except GeocoderTimedOut:
            print(f"Geocoding timed out on attempt {attempt + 1}. Retrying...")
            time.sleep(2)  # 2초 대기 후 재시도
        except GeocoderServiceError as e:
            print(f"Geocoding service error: {e}")
            return None, None  # 서비스 오류 시, 결과 반환하지 않음
        except Exception as e:
            print(f"Unexpected error: {e}")
            return None, None  # 예상치 못한 오류 시, 결과 반환하지 않음
    return None, None  # 재시도 후에도 실패하면 None 반환

# 위도와 경도 컬럼 추가
latitudes = []
longitudes = []

for index, row in df.iterrows():
    if pd.notna(row['장소']):
        # 쏘카존 지점으로 지오코딩 시도
        lat, lon = geocode_address(row['장소'])
        if lat is None or lon is None:
            # 실패하면 도로명 주소로 지오코딩 시도
            if pd.notna(row['도로명 주소']):
                lat, lon = geocode_address(row['도로명 주소'])
    else:
        lat, lon = None, None
    
    latitudes.append(lat)
    longitudes.append(lon)

df['위도'] = latitudes
df['경도'] = longitudes

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)


In [ ]:
import pandas as pd

# 두 개의 CSV 파일 읽기
df1 = pd.read_csv('socar_locations_combined_unique_final.csv', encoding='utf-8-sig')
df2 = pd.read_csv('socar_locations_geo_all.csv', encoding='utf-8-sig')

# 두 데이터프레임 합치기
df_combined = pd.concat([df1, df2])

# 열 이름을 적절히 수정
df_combined.columns = ['장소', '도로명 주소', '위도', '경도']

# 1. 열 1('장소')에서 중복 제거 (위도와 경도가 존재하는 행을 우선으로 남김)
df_combined = df_combined.sort_values(by=['위도', '경도'], na_position='last')
df_combined = df_combined.drop_duplicates(subset=['장소'], keep='first')

# 2. 열 2('도로명 주소')에서 중복 제거 (위도와 경도가 존재하는 행을 우선으로 남김)
df_combined = df_combined.sort_values(by=['위도', '경도'], na_position='last')
df_combined = df_combined.drop_duplicates(subset=['도로명 주소'], keep='first')

# 결과 저장 (소수점 자릿수를 제한하지 않고 저장)
df_combined.to_csv('socar_locations_cleaned_last2.csv', index=False, encoding='utf-8-sig', float_format='%.15g')

print("중복 제거 및 정리 완료. 결과는 'socar_locations_cleaned_last2.csv'에 저장되었습니다.")


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\lastlast.csv'
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\lastlast_result.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder", timeout=10)  # timeout 값을 10초로 설정

# 지오코딩 함수 정의
def geocode_address(address, retries=5):
    for attempt in range(retries):
        try:
            location = geolocator.geocode(address)
            if location:
                return location.latitude, location.longitude
        except GeocoderTimedOut:
            print(f"Geocoding timed out on attempt {attempt + 1}. Retrying...")
            time.sleep(2)  # 2초 대기 후 재시도
        except GeocoderServiceError as e:
            print(f"Geocoding service error: {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")
    return None, None  # 재시도 후에도 실패하면 None 반환

# 위도와 경도 컬럼 추가
latitudes = []
longitudes = []

for index, row in df.iterrows():
    lat, lon = None, None
    if pd.notna(row['장소']):
        # 쏘카존 지점으로 지오코딩 시도
        lat, lon = geocode_address(row['장소'])
    if (lat is None or lon is None) and pd.notna(row['도로명 주소']):
        # 실패하면 도로명 주소로 지오코딩 시도
        lat, lon = geocode_address(row['도로명 주소'])
    
    latitudes.append(lat)
    longitudes.append(lon)

df['위도'] = latitudes
df['경도'] = longitudes

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_combined_unique_final.csv'
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_geocoded_empty_only.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder", timeout=10)  # timeout 값을 10초로 설정

# 지오코딩 함수 정의
def geocode_address(address, retries=5):
    for attempt in range(retries):
        try:
            location = geolocator.geocode(address)
            if location:
                return location.latitude, location.longitude
        except GeocoderTimedOut:
            print(f"Geocoding timed out on attempt {attempt + 1}. Retrying...")
            time.sleep(2)  # 2초 대기 후 재시도
        except GeocoderServiceError as e:
            print(f"Geocoding service error: {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")
    return None, None  # 재시도 후에도 실패하면 None 반환

# 위도와 경도 컬럼 추가
latitudes = []
longitudes = []

for index, row in df.iterrows():
    lat, lon = None, None
    
    # 세 번째 열(예: "세 번째 열 이름")이 비어 있을 때
    if pd.isna(row['세 번째 열 이름']):
        # 첫 번째 열(예: "장소")로 지오코딩 시도
        if pd.notna(row['장소']):
            lat, lon = geocode_address(row['장소'])
        
        # 첫 번째 열로 지오코딩 실패 시 두 번째 열(예: "도로명 주소")로 지오코딩 시도
        if (lat is None or lon is None) and pd.notna(row['도로명 주소']):
            lat, lon = geocode_address(row['도로명 주소'])
    
    # 빈 곳만 처리하기 위해 빈 곳만 리스트에 추가
    if lat is not None and lon is not None:
        latitudes.append(lat)
        longitudes.append(lon)
    else:
        latitudes.append(None)
        longitudes.append(None)

# 빈 곳만 포함된 새로운 데이터프레임 생성
df_empty_geocoded = df[pd.isna(df['위도'])].copy()
df_empty_geocoded['위도'] = latitudes
df_empty_geocoded['경도'] = longitudes

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_empty_geocoded[['위도', '경도']].to_csv(output_file_path, index=False, encoding='utf-8')

print("빈 곳에 대한 지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)


In [ ]:
import pandas as pd

# CSV 파일 경로 설정
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_cleaned_last.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
    print("UTF-8 인코딩으로 파일을 성공적으로 읽었습니다.")
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')
    print("CP949 인코딩으로 파일을 성공적으로 읽었습니다.")

# '도로명 주소' 열이 존재하는지 확인
if '도로명 주소' not in df.columns:
    raise ValueError("데이터프레임에 '도로명 주소' 열이 존재하지 않습니다.")

# 지역명 추출 함수 정의
def extract_region(address):
    if pd.isna(address):
        return None
    # ' '로 나눈 첫 번째 단어 반환
    region = address.split(' ')[0]
    return region

# 새로운 열에 지역명 저장
df['지역명'] = df['도로명 주소'].apply(extract_region)

# 지역명 열만 추출하여 출력
print(df['지역명'])


In [ ]:
import pandas as pd

# CSV 파일 경로 설정
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_cleaned_last.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
    print("UTF-8 인코딩으로 파일을 성공적으로 읽었습니다.")
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')
    print("CP949 인코딩으로 파일을 성공적으로 읽었습니다.")

# '도로명 주소' 열이 존재하는지 확인
if '도로명 주소' not in df.columns:
    raise ValueError("데이터프레임에 '도로명 주소' 열이 존재하지 않습니다.")

# 지역명 추출 함수 정의
def extract_region(address):
    if pd.isna(address):
        return None
    # ' '로 나눈 첫 번째 단어 반환
    region = address.split(' ')[0]
    return region

# 새로운 열에 지역명 저장
df['지역명'] = df['도로명 주소'].apply(extract_region)

# 앞뒤 공백 제거
df['지역명'] = df['지역명'].str.strip()

# 모든 `지역명`을 인덱스 없이 세로로 출력
print("추출된 모든 지역명 (인덱스 없이, 공백 제거):")
print(df['지역명'].to_string(index=False))  # 인덱스 없이 출력

# `지역명` 열만 포함된 새로운 CSV 파일로 저장
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_region_only.csv'
df[['지역명']].to_csv(output_file_path, index=False, encoding='utf-8')
print(f"지역명 열만 포함된 CSV 파일이 저장되었습니다: {output_file_path}")

# `지역명` 열을 텍스트 파일로 저장
text_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_region_only.txt'
df['지역명'].to_csv(text_file_path, index=False, header=False, encoding='utf-8')
print(f"지역명 열만 포함된 텍스트 파일이 저장되었습니다: {text_file_path}")
